## Exploratory Data Analysis

#### Objective

The objective is to predict the number of shares (popularity) on social media platforms for the news articles using the dataset provided. In your submission, you should evaluate at least 3 suitable models for predicting the number of shares.

#### Overall Evaluation

Once you have identified the most suitable model, find a way to compare the models against each other and explain which of these models is more suitable for the task.

In [1]:
#import modules
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error, r2_score
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV

## Preliminary Data Exploration

In [3]:
#load dataset
df = pd.read_csv('./data/mini_project_1_data.csv')

df.head()

,ID,URL,timedelta,weekday,shares,n_tokens_title,n_tokens_content,n_unique_tokens,n_non_stop_words,n_non_stop_unique_tokens,...,num_keywords,kw_min_min,kw_max_min,kw_avg_min,kw_min_max,kw_max_max,kw_avg_max,kw_min_avg,kw_max_avg,kw_avg_avg
0,0,http://mashable.com/2013/01/07/amazon-instant-...,731.0,monday,593,12.0,219.0,0.663594,1.0,0.815385,...,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1,http://mashable.com/2013/01/07/ap-samsung-spon...,731.0,monday,711,9.0,255.0,0.604743,1.0,0.791946,...,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2,http://mashable.com/2013/01/07/apple-40-billio...,731.0,monday,1500,9.0,211.0,0.575130,1.0,0.663866,...,6.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,3,http://mashable.com/2013/01/07/astronaut-notre...,731.0,monday,1200,9.0,531.0,0.503788,1.0,0.665635,...,7.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,4,http://mashable.com/2013/01/07/att-u-verse-apps/,731.0,monday,505,13.0,1072.0,0.415646,1.0,0.540890,...,7.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [8]:
df.URL.head(100)

0     http://mashable.com/2013/01/07/amazon-instant-...
1     http://mashable.com/2013/01/07/ap-samsung-spon...
2     http://mashable.com/2013/01/07/apple-40-billio...
3     http://mashable.com/2013/01/07/astronaut-notre...
4      http://mashable.com/2013/01/07/att-u-verse-apps/
                            ...                        
95       http://mashable.com/2013/01/09/ces-2013-comic/
96    http://mashable.com/2013/01/09/ces-2013-show-f...
97    http://mashable.com/2013/01/09/ces-challenge-n...
98    http://mashable.com/2013/01/09/chesapeake-ligh...
99    http://mashable.com/2013/01/09/crowdrise-edwar...
Name: URL, Length: 100, dtype: object

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35680 entries, 0 to 35679
Data columns (total 30 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   ID                         35680 non-null  int64  
 1   URL                        35680 non-null  object 
 2   timedelta                  35680 non-null  float64
 3   weekday                    35680 non-null  object 
 4   shares                     35680 non-null  int64  
 5   n_tokens_title             35680 non-null  float64
 6   n_tokens_content           35680 non-null  float64
 7   n_unique_tokens            35680 non-null  float64
 8   n_non_stop_words           35680 non-null  float64
 9   n_non_stop_unique_tokens   35680 non-null  float64
 10  num_hrefs                  34959 non-null  float64
 11  num_self_hrefs             34959 non-null  float64
 12  num_imgs                   34261 non-null  float64
 13  num_videos                 18925 non-null  flo

In [6]:
print(df.nunique())
print()
print("Check Categorical and Boolean Column unique values:")
print(f"{df.weekday.name}:{df.weekday.unique()}")
print(f"{df.n_tokens_title.name}:{df.n_tokens_title.unique()}")
print(f"{df.data_channel.name}:{df.data_channel.unique()}")
print(f"{df.num_keywords.name}:{df.num_keywords.unique()}")


ID                           35680
URL                          35680
timedelta                      724
weekday                          7
shares                        1418
n_tokens_title                  20
n_tokens_content              2324
n_unique_tokens              25197
n_non_stop_words              1429
n_non_stop_unique_tokens     21345
num_hrefs                      128
num_self_hrefs                  57
num_imgs                        89
num_videos                      46
n_comments                     834
average_token_length         27595
data_channel                     6
self_reference_min_shares     1231
self_reference_max_shares     1119
self_reference_avg_shares     7853
num_keywords                    10
kw_min_min                      22
kw_max_min                    1064
kw_avg_min                   15949
kw_min_max                    1006
kw_max_max                      31
kw_avg_max                   28284
kw_min_avg                   14686
kw_max_avg          

In [9]:
df.describe()

,ID,timedelta,shares,n_tokens_title,n_tokens_content,n_unique_tokens,n_non_stop_words,n_non_stop_unique_tokens,num_hrefs,num_self_hrefs,...,num_keywords,kw_min_min,kw_max_min,kw_avg_min,kw_min_max,kw_max_max,kw_avg_max,kw_min_avg,kw_max_avg,kw_avg_avg
count,35680.000000,35680.000000,35680.000000,35680.000000,35680.000000,35680.000000,35680.000000,35680.000000,34959.000000,34959.000000,...,35680.000000,35680.000000,35680.000000,35680.000000,35680.000000,35680.000000,35680.000000,35680.000000,35680.000000,35680.000000
mean,19823.468918,354.523234,3402.054961,10.395432,545.995123,0.530461,0.970263,0.672738,10.866358,3.282531,...,7.220095,26.169703,1142.296851,310.297441,13533.498851,752318.562220,259143.787814,1115.670961,5637.977838,3130.799667
std,11452.292291,214.301086,11934.878541,2.114600,469.550052,0.136935,0.169862,0.153881,11.260971,3.834065,...,1.909604,69.705812,3844.829034,616.961660,57623.779316,214391.922961,135080.424060,1136.741358,6026.965362,1311.012948
min,0.000000,8.000000,4.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,1.000000,-1.000000,0.000000,-1.000000,0.000000,0.000000,0.000000,-1.000000,0.000000,0.000000
25%,9890.750000,163.000000,946.000000,9.000000,247.000000,0.471030,1.000000,0.625912,4.000000,1.000000,...,6.000000,-1.000000,445.000000,141.440476,0.000000,843300.000000,172595.650000,0.000000,3559.646658,2379.762290
50%,19801.500000,339.000000,1400.000000,10.000000,410.000000,0.539011,1.000000,0.690323,8.000000,3.000000,...,7.000000,-1.000000,658.000000,235.428571,1400.000000,843300.000000,244133.333333,1020.667532,4348.584651,2867.991456
75%,29788.250000,543.000000,2800.000000,12.000000,716.000000,0.608203,1.000000,0.754036,14.000000,4.000000,...,9.000000,4.000000,1000.000000,357.000000,7800.000000,843300.000000,330980.000000,2058.093423,6015.434738,3595.530145
max,39643.000000,731.000000,843300.000000,23.000000,8474.000000,1.000000,1.000000,1.000000,304.000000,116.000000,...,10.000000,377.000000,298400.000000,42827.857143,843300.000000,843300.000000,843300.000000,3610.124972,298400.000000,43567.659946


In [ ]:
# Counting number of negative values in each kw feature
kw_feature_list = ['kw_min_min', 'kw_max_min', 'kw_avg_min', 'kw_min_max', 'kw_max_max', 'kw_avg_max', 'kw_min_avg', 'kw_max_avg', 'kw_avg_avg']
for feature in kw_feature_list:
    count = (df[feature] < 0).sum()
    print(f"{feature}: {count} negative values")


kw_min_min: 20660 negative values
kw_max_min: 0 negative values
kw_avg_min: 767 negative values
kw_min_max: 0 negative values
kw_max_max: 0 negative values
kw_avg_max: 0 negative values
kw_min_avg: 6 negative values
kw_max_avg: 0 negative values
kw_avg_avg: 0 negative values


In [ ]:
# Investigating negative values for kw_avg_min
df[df['kw_avg_min'] < 0]

,ID,URL,timedelta,weekday,shares,n_tokens_title,n_tokens_content,n_unique_tokens,n_non_stop_words,n_non_stop_unique_tokens,...,num_keywords,kw_min_min,kw_max_min,kw_avg_min,kw_min_max,kw_max_max,kw_avg_max,kw_min_avg,kw_max_avg,kw_avg_avg
15687,17408,http://mashable.com/2013/12/11/youtube-profess...,393.0,wednesday,446,8.0,1284.0,0.460033,1.0,0.618438,...,5.0,-1.0,1.59,-0.482000,210300.0,843300.0,591160.000000,2855.269091,5628.965321,3908.614194
15917,17663,http://mashable.com/2013/12/17/facebook-video-...,387.0,tuesday,902,9.0,324.0,0.749153,1.0,0.880597,...,4.0,-1.0,1.59,-0.352500,210300.0,843300.0,584475.000000,2883.592372,5671.272007,3750.179825
16037,17798,http://mashable.com/2013/12/19/google-transpar...,385.0,thursday,531,11.0,158.0,0.707006,1.0,0.810000,...,1.0,-1.0,0.00,-1.000000,843300.0,843300.0,843300.000000,3472.638769,3472.638769,3472.638769
16101,17869,http://mashable.com/2013/12/20/most-searched-m...,384.0,friday,731,9.0,322.0,0.599359,1.0,0.747475,...,2.0,-1.0,0.00,-1.000000,210300.0,843300.0,526800.000000,2954.871610,3477.720952,3216.296281
16236,18011,http://mashable.com/2013/12/24/video-game-writ...,379.0,wednesday,1100,10.0,804.0,0.412204,1.0,0.556237,...,1.0,-1.0,0.00,-1.000000,843300.0,843300.0,843300.000000,3477.320629,3477.320629,3477.320629
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35639,39600,http://mashable.com/2014/12/26/marshall-woburn...,9.0,tuesday,2000,11.0,1166.0,0.508254,1.0,0.736842,...,3.0,-1.0,1.59,-0.136667,690400.0,843300.0,741366.666667,3104.928541,5051.467451,3847.263104
35650,39611,http://mashable.com/2014/12/26/sony-playstatio...,9.0,tuesday,4600,12.0,270.0,0.604651,1.0,0.670213,...,4.0,-1.0,0.00,-1.000000,45100.0,843300.0,411925.000000,2141.095781,3385.393320,2739.516721
35665,39627,http://mashable.com/2014/12/27/extremist-leade...,9.0,tuesday,1200,14.0,874.0,0.480702,1.0,0.711538,...,6.0,-1.0,0.00,-1.000000,0.0,843300.0,333833.333333,0.000000,3385.393320,1996.065900
35668,39631,http://mashable.com/2014/12/27/japan-elderly-d...,9.0,tuesday,914,13.0,746.0,0.465306,1.0,0.735849,...,5.0,-1.0,0.00,-1.000000,23100.0,843300.0,281640.000000,1558.755814,3385.393320,2314.316345


In [25]:
# Investigating negative values for kw_min_avg
df[df['kw_min_avg'] < 0]

,ID,URL,timedelta,weekday,shares,n_tokens_title,n_tokens_content,n_unique_tokens,n_non_stop_words,n_non_stop_unique_tokens,...,num_keywords,kw_min_min,kw_max_min,kw_avg_min,kw_min_max,kw_max_max,kw_avg_max,kw_min_avg,kw_max_avg,kw_avg_avg
32463,36070,http://mashable.com/2014/11/03/bill-clinton-ph...,66.0,monday,1900,9.0,153.0,0.756579,1.0,0.952941,...,8.0,-1.0,758.0,125.500000,0.0,843300.0,277700.000000,-1.0,3437.392296,1380.847901
32465,36072,http://mashable.com/2014/11/03/bra-facts/,65.0,tuesday,5300,11.0,1110.0,0.447419,1.0,0.593558,...,9.0,-1.0,730.0,184.888889,0.0,843300.0,249677.777778,-1.0,3433.817566,1747.204777
34633,38475,http://mashable.com/2014/12/08/sony-hack-unpre...,29.0,wednesday,706,14.0,1162.0,0.472537,1.0,0.674855,...,10.0,-1.0,330.0,81.750000,0.0,843300.0,252033.000000,-1.0,3399.331393,1428.590860
34655,38501,http://mashable.com/2014/12/09/amazon-supreme-...,29.0,wednesday,670,16.0,364.0,0.635569,1.0,0.695473,...,9.0,-1.0,151.0,20.714286,0.0,843300.0,222222.222222,-1.0,3399.331393,1553.928030
34680,38529,http://mashable.com/2014/12/09/facebook-year-i...,29.0,wednesday,2800,14.0,604.0,0.498305,1.0,0.641558,...,7.0,-1.0,679.0,112.333333,0.0,843300.0,317928.571429,-1.0,3399.331393,1665.071561
35171,39083,http://mashable.com/2014/12/17/cats-vs-christm...,20.0,friday,994,12.0,0.0,0.000000,0.0,0.000000,...,8.0,-1.0,1600.0,291.750000,0.0,843300.0,236655.000000,-1.0,3393.367028,1717.644684


In [11]:
df.shape

(35680, 30)

In [12]:
df.isnull().sum()

ID                               0
URL                              0
timedelta                        0
weekday                          0
shares                           0
n_tokens_title                   0
n_tokens_content                 0
n_unique_tokens                  0
n_non_stop_words                 0
n_non_stop_unique_tokens         0
num_hrefs                      721
num_self_hrefs                 721
num_imgs                      1419
num_videos                   16755
n_comments                       0
average_token_length             0
data_channel                  5462
self_reference_min_shares      721
self_reference_max_shares      721
self_reference_avg_shares      721
num_keywords                     0
kw_min_min                       0
kw_max_min                       0
kw_avg_min                       0
kw_min_max                       0
kw_max_max                       0
kw_avg_max                       0
kw_min_avg                       0
kw_max_avg          

In [ ]:
print(f"IDs duplicated: {df['ID'].duplicated().sum()}")
duplicated_df = df[df["ID"].duplicated(keep=False)].sort_values(by="ID")
duplicated_df.head(25)

IDs duplicated: 0
URLs duplicated: 0


,ID,URL,timedelta,weekday,shares,n_tokens_title,n_tokens_content,n_unique_tokens,n_non_stop_words,n_non_stop_unique_tokens,...,num_keywords,kw_min_min,kw_max_min,kw_avg_min,kw_min_max,kw_max_max,kw_avg_max,kw_min_avg,kw_max_avg,kw_avg_avg


In [15]:
print(f"URLs duplicated: {df['URL'].duplicated().sum()}")
duplicated_df = df[df["URL"].duplicated(keep=False)].sort_values(by="URL")
duplicated_df.head(25)

URLs duplicated: 0


,ID,URL,timedelta,weekday,shares,n_tokens_title,n_tokens_content,n_unique_tokens,n_non_stop_words,n_non_stop_unique_tokens,...,num_keywords,kw_min_min,kw_max_min,kw_avg_min,kw_min_max,kw_max_max,kw_avg_max,kw_min_avg,kw_max_avg,kw_avg_avg


#### Data Structure, Assumptions and Anomalies

* The dataset has 35680 rows and 30 columns.
* ID is assumed to be the primary key for this dataset. URL was also investigated as a potential unique key. But there are no cases of duplication for both ID and URL so we will keep ID as the assumed primary key.
* There are null values num_hrefs, num_self_hrefs num_imgs, num_videos, self_reference_min_shares, self_reference_max_shares and self_reference_avg_shares. We will assume these null values are indicative of an absence of the respective content for those articles so these values will be replaced with 0.
* The data channel contains categorical values for the category of channel the article is written for. Therefore we will assume those with null values falls under no category and we will rename it to None.
* There are negative values for kw_min_min and kw_min_avg detected. For these two features as it is not possible for share count to be negative, we will need to clean this by replacing the negative values with 0.


## Data Cleaning

In [26]:
df_cleaned = df.copy()

# Replace selected features with null values with 0s
replace_null_feature_list = ['num_hrefs', 'num_self_hrefs', 'num_imgs', 'num_videos', 'self_reference_min_shares', 'self_reference_max_shares', 'self_reference_avg_shares']
for feature in replace_null_feature_list:
    df_cleaned[feature] = df_cleaned[feature].fillna(0)
    print(f"{feature} NaN Count: {df_cleaned[feature].isnull().sum()}") # Check that null values are replaced

# Replace null values in data channel with none
df_cleaned['data_channel'] = df_cleaned['data_channel'].fillna('none')
print(f"data_channel unique values: {df_cleaned['data_channel'].unique()}") # Check that null values are replaced

# Replace negative values in kw features with 0 using clip
kw_feature_list = ['kw_avg_min', 'kw_min_avg']
for feature in kw_feature_list:
    df[feature] = df[feature].clip(lower=0)
for feature in kw_feature_list: # check that negative values have been replaced
    count = (df[feature] < 0).sum()
    print(f"{feature}: {count} negative values")


num_hrefs NaN Count: 0
num_self_hrefs NaN Count: 0
num_imgs NaN Count: 0
num_videos NaN Count: 0
self_reference_min_shares NaN Count: 0
self_reference_max_shares NaN Count: 0
self_reference_avg_shares NaN Count: 0
data_channel unique values: ['entertainment' 'business' 'technology' 'lifestyle' 'world' 'none'
 'social_media']
kw_avg_min: 0 negative values
kw_min_avg: 0 negative values
